##**Silver Data Transformation**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("abfss://bronze@netflixazuredatalake.dfs.core.windows.net/netflix_titles")

In [0]:
df.display()
    

In [0]:
df = df.fillna({"duration_minutes": 0, "duration_seasons": 1})

In [0]:
df = df.withColumn("duration_minutes",col("duration_minutes").cast(IntegerType()))\
    .withColumn("duration_seasons",col("duration_seasons").cast(IntegerType()))
   

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("new_title",split(col("title"),':')[0])\
    .withColumn("new_rating",split(col("rating"),'-')[0])

In [0]:
df=df.withColumn('New_Flag',when(col('Type')=='Movie',1)\
    .when(col('Type')=='TV Show',2)\
    .otherwise(0))

In [0]:
from pyspark.sql.window import Window

In [0]:
df = df.withColumn("den_rank", dense_rank().over(Window.orderBy(col("duration_minutes").desc())))
    

In [0]:
display(df)

In [0]:
from pyspark.sql.functions import count

In [0]:
df_vis = df.groupBy("type").agg(count("*").alias("Total_Count"))
display(df_vis)

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.mode("overwrite")\
    .format("delta")\
    .option("path","abfss://silver@netflixazuredatalake.dfs.core.windows.net/netflix_titles")\
    .save()